In [1]:
from pathlib import Path
import sys
import torch


def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for c in candidates:
        if (c / "AAAI24_GARCH_NN_Reproduction").exists() and (c / "dataset").exists():
            return c
    raise RuntimeError("Cannot locate project root from current working directory")


def describe_gpu_and_pick_device():
    if not torch.cuda.is_available():
        print("CUDA available: False")
        return "cpu"

    device_idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_idx)
    total_gb = props.total_memory / (1024 ** 3)
    print("CUDA available: True")
    print(f"GPU: {props.name} (index={device_idx})")
    print(f"Total VRAM: {total_gb:.2f} GB")
    print(f"CUDA capability: {props.major}.{props.minor}")
    return f"cuda:{device_idx}"


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DEFAULT_DEVICE = describe_gpu_and_pick_device()

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")
print(f"Selected device: {DEFAULT_DEVICE}")

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU (index=0)
Total VRAM: 4.00 GB
CUDA capability: 8.6
Project root: D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT
Python executable: d:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\.venv\Scripts\python.exe
Selected device: cuda:0


In [2]:
# Run mode: smoke (1 dataset, 1 epoch) or full (all datasets, 60 epochs)
mode = "full"
SMOKE_DATASET = "VN30_INDEX"

if mode not in {"smoke", "full"}:
    raise ValueError("mode must be 'smoke' or 'full'")

print(f"Mode: {mode}")
if mode == "smoke":
    print(f"Smoke dataset: {SMOKE_DATASET}")

Mode: full


In [3]:
DATASET_DIR = PROJECT_ROOT / "dataset"
ALL_DATASETS = [
    #"VN30_INDEX",
    #"DAX_40",
    #"EuroNext_100",
    #"IBEX_35",
    "KOSPI_index",
    "Nikkei_225",
    "SMI",
    "snp500",
    "VN_INDEX"

]

selected_datasets = [SMOKE_DATASET] if mode == "smoke" else ALL_DATASETS
DATASET_FILES = [DATASET_DIR / f"{name}.csv" for name in selected_datasets]
missing_files = [str(path) for path in DATASET_FILES if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Dataset file(s) not found: {missing_files}")

print(f"Selected {len(DATASET_FILES)} dataset(s):")
for path in DATASET_FILES:
    print(" -", path.name)

Selected 5 dataset(s):
 - KOSPI_index.csv
 - Nikkei_225.csv
 - SMI.csv
 - snp500.csv
 - VN_INDEX.csv


In [4]:
import importlib
import AAAI24_GARCH_NN_Reproduction.core.data_processor as dp_module
import AAAI24_GARCH_NN_Reproduction.experiments.run_benchmark as run_benchmark_module
import AAAI24_GARCH_NN_Reproduction.models.dl_baselines as dl_module

# Reload all modified modules to pick up latest changes
dp_module = importlib.reload(dp_module)
dl_module = importlib.reload(dl_module)
run_benchmark_module = importlib.reload(run_benchmark_module)

# Import functions from reloaded modules
from AAAI24_GARCH_NN_Reproduction.core.data_processor import create_sliding_windows
run_benchmark = run_benchmark_module.run_benchmark

SEQ_LEN = 60
EPOCHS = 1 if mode == "smoke" else 60
BATCH_SIZE = 128
DEVICE = DEFAULT_DEVICE
NUM_WORKERS = 0  # Set to 0 to avoid DataLoader multiprocessing issues on Windows
LOG_PROGRESS = True

# ========== MULTI-HORIZON MODE CONFIGURATION ==========
# Choose forecasting strategy:
#   "rolling" (default): Train horizon=1 only, use rolling index to extract multi-horizon
#   "one_shot": Train all 21 horizons simultaneously in one model output
MULTI_HORIZON_MODE = "rolling"  # Change to "one_shot" for one-shot multi-horizon (WIP)
print(f"Multi-Horizon Mode: {MULTI_HORIZON_MODE}")

if str(DEVICE).startswith("cuda"):
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")

RESULTS_DIR = PROJECT_ROOT / "AAAI24_GARCH_NN_Reproduction" / "experiments" / "results_6"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "AAAI24_Baselines"
OUTPUT_PREDICTIONS_CSV = RESULTS_DIR / f"{MODEL_NAME}_prediction.csv"

# Hyperparameter tuning config (Tier 1 & 2 priorities)
TUNING_CONFIG = {
    "learning_rate": 1e-2,          # [Tier 1] Main optimizer step size
    "lr_factor": 0.5,               # [Tier 2] LR decay multiplier
    "lr_patience": 8,               # [Tier 1] Epochs before LR decay
    "early_stopping_patience": 15,  # [Tier 1] Epochs before early stopping
    "min_lr": 1e-5,                 # [Tier 2] Minimum learning rate floor
}

# Update run_benchmark with tuning config and multi-horizon mode
run_benchmark_module.HYPERPARAMETER_CONFIG.update(TUNING_CONFIG)
run_benchmark_module.MULTI_HORIZON_MODE = MULTI_HORIZON_MODE  # ← Pass mode to run_benchmark

print(f"Training Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Device: {DEVICE}")
print(f"  Num workers: {NUM_WORKERS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Sequence length: {SEQ_LEN}")
print(f"  Output CSV: {OUTPUT_PREDICTIONS_CSV}")
print(f"  Multi-Horizon Mode: {MULTI_HORIZON_MODE}")
print(f"\nTuning Configuration (applied):")
for key, val in TUNING_CONFIG.items():
    print(f"  {key}: {val}")

Multi-Horizon Mode: rolling
Training Configuration:
  Epochs: 60
  Device: cuda:0
  Num workers: 0
  Batch size: 128
  Sequence length: 60
  Output CSV: D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\AAAI24_GARCH_NN_Reproduction\experiments\results_6\AAAI24_Baselines_prediction.csv
  Multi-Horizon Mode: rolling

Tuning Configuration (applied):
  learning_rate: 0.01
  lr_factor: 0.5
  lr_patience: 8
  early_stopping_patience: 15
  min_lr: 1e-05


In [5]:
# Data Processing and Splitting Configuration
# ================================================

# Split mode: "ratio" (8:1:1 proportional split) or "fixed_counts" (predefined table for 2010-2025)
SPLIT_MODE = "fixed_counts"  # Change to "ratio" to use 8:1:1 proportional split

# Predefined split counts for fixed_counts mode (Train, Val, Test) - data range 2010-2025
# These MUST match the SPLIT_COUNTS specification exactly
SPLIT_COUNTS = {
    "VN30_INDEX": (1971, 1096, 925),
    "VN_INDEX": (1964, 1103, 925),
    "DAX_40": (1983, 1104, 972),
    "EuroNext_100": (2005, 1115, 979),
    "IBEX_35": (1815, 1362, 923),
    "KOSPI_index": (1944, 1109, 881),
    "SMI": (1987, 1135, 901),
    "snp500": (1988, 1109, 927),
    "Nikkei_225": (1677, 1174, 1062),
}

# Date range filtering (only used for filtering, not in ratio mode)
# If DATE_START/DATE_END are None, no filtering is applied (uses all available data)
DATE_START = "2010-01-01"  # Format: YYYY-MM-DD or None
DATE_END = "2025-12-31"    # Format: YYYY-MM-DD or None

print(f"Data Split Configuration:")
print(f"  Split mode: {SPLIT_MODE}")
print(f"  Date range: {DATE_START or 'from start'} to {DATE_END or 'to end'}")

if SPLIT_MODE == "fixed_counts":
    print(f"\n  ✓ Using FIXED SPLIT COUNTS (Train, Val, Test):")
    for dataset_name, (train_cnt, val_cnt, test_cnt) in SPLIT_COUNTS.items():
        total = train_cnt + val_cnt + test_cnt
        print(f"    {dataset_name:20s}: Train={train_cnt:4d}, Val={val_cnt:4d}, Test={test_cnt:4d} (Total={total:4d})")
    
    # Verify all selected datasets have split counts defined
    missing_datasets = [d for d in selected_datasets if d not in SPLIT_COUNTS]
    if missing_datasets:
        print(f"\n  ✗ ERROR: Missing split counts for: {missing_datasets}")
        raise ValueError(f"Missing SPLIT_COUNTS for datasets: {missing_datasets}")
    else:
        print(f"\n  ✓ All {len(selected_datasets)} selected dataset(s) have valid split counts defined")
elif SPLIT_MODE == "ratio":
    print(f"  (Using 80%-10%-10% proportional split)")

Data Split Configuration:
  Split mode: fixed_counts
  Date range: 2010-01-01 to 2025-12-31

  ✓ Using FIXED SPLIT COUNTS (Train, Val, Test):
    VN30_INDEX          : Train=1971, Val=1096, Test= 925 (Total=3992)
    VN_INDEX            : Train=1964, Val=1103, Test= 925 (Total=3992)
    DAX_40              : Train=1983, Val=1104, Test= 972 (Total=4059)
    EuroNext_100        : Train=2005, Val=1115, Test= 979 (Total=4099)
    IBEX_35             : Train=1815, Val=1362, Test= 923 (Total=4100)
    KOSPI_index         : Train=1944, Val=1109, Test= 881 (Total=3934)
    SMI                 : Train=1987, Val=1135, Test= 901 (Total=4023)
    snp500              : Train=1988, Val=1109, Test= 927 (Total=4024)
    Nikkei_225          : Train=1677, Val=1174, Test=1062 (Total=3913)

  ✓ All 5 selected dataset(s) have valid split counts defined


In [6]:
import pandas as pd


# Verify SPLIT_COUNTS are properly configured
print("="*70)
print("SPLIT_COUNTS VERIFICATION")
print("="*70)
print(f"Split mode: {SPLIT_MODE}")
if SPLIT_MODE == "fixed_counts":
    print(f"\nChecking SPLIT_COUNTS match:")
    for original_name in selected_datasets:
        if original_name in SPLIT_COUNTS:
            train, val, test = SPLIT_COUNTS[original_name]
            total = train + val + test
            print(f"  ✓ {original_name:20s}: Train={train:4d}, Val={val:4d}, Test={test:4d} (Total={total:4d})")
        else:
            print(f"  ✗ {original_name:20s}: NOT FOUND IN SPLIT_COUNTS!")
            raise KeyError(f"Dataset '{original_name}' not in SPLIT_COUNTS")
    print(f"\n  ✓ All {len(selected_datasets)} dataset(s) verified in SPLIT_COUNTS")

print("="*70)
print()

# Determine number of seeds based on mode
NUM_SEEDS = 1 if mode == "smoke" else 5
print(f"Number of seeds: {NUM_SEEDS} (mode={mode})")
print()


def run_predictions(dataset_files):
    prediction_frames = []

    for dataset_file in dataset_files:
        print(f"Running benchmark: {dataset_file.name}")
        # Pass output_csv pointing to results_5 folder so predictions.csv saves there
        output_model_results = RESULTS_DIR / f"model_results.csv"
        run_benchmark(
            dataset_dir=dataset_file,
            seq_len=SEQ_LEN,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            output_csv=str(output_model_results),  # ← This controls where predictions.csv will be saved
            device=DEVICE,
            num_workers=NUM_WORKERS,
            log_progress=LOG_PROGRESS,
            split_mode=SPLIT_MODE,
            date_start=DATE_START,
            date_end=DATE_END,
            num_seeds=NUM_SEEDS,  # ← Use 1 seed for smoke, 5 for full
        )

        # predictions.csv is saved in the same directory as output_csv
        per_dataset_predictions = RESULTS_DIR / "predictions.csv"
        if not per_dataset_predictions.exists():
            print(f"ERROR: Expected file not found at {per_dataset_predictions}")
            print(f"Files in {RESULTS_DIR}:")
            import os
            if RESULTS_DIR.exists():
                for f in sorted(os.listdir(RESULTS_DIR)):
                    print(f"  - {f}")
            raise FileNotFoundError(f"Missing: {per_dataset_predictions}")

        prediction_frames.append(pd.read_csv(per_dataset_predictions))

    if not prediction_frames:
        return pd.DataFrame(
            columns=[
                "time",
                "dataset",
                "model",
                "horizon",
                "True_Volatility",
                "Pred_Volatility",
                "return_1_day",
            ]
        )

    return pd.concat(prediction_frames, ignore_index=True)


predictions_df = run_predictions(DATASET_FILES)
required_columns = [
    "time",
    "dataset",
    "model",
    "horizon",
    "True_Volatility",
    "Pred_Volatility",
    "return_1_day",
]
predictions_df = predictions_df[required_columns]
predictions_df.to_csv(OUTPUT_PREDICTIONS_CSV, index=False)

print(f"\nSaved final aggregated predictions: {OUTPUT_PREDICTIONS_CSV}")
print(f"Total rows: {len(predictions_df)}")
display(predictions_df.head())

SPLIT_COUNTS VERIFICATION
Split mode: fixed_counts

Checking SPLIT_COUNTS match:
  ✓ KOSPI_index         : Train=1944, Val=1109, Test= 881 (Total=3934)
  ✓ Nikkei_225          : Train=1677, Val=1174, Test=1062 (Total=3913)
  ✓ SMI                 : Train=1987, Val=1135, Test= 901 (Total=4023)
  ✓ snp500              : Train=1988, Val=1109, Test= 927 (Total=4024)
  ✓ VN_INDEX            : Train=1964, Val=1103, Test= 925 (Total=3992)

  ✓ All 5 dataset(s) verified in SPLIT_COUNTS

Number of seeds: 5 (mode=full)

Running benchmark: KOSPI_index.csv
[19:17:04] Benchmark started | device=cuda:0 | datasets=1 | seeds=5 | horizons=[1, 3, 5, 10, 21]
[19:17:04] Dataset start: KOSPI_index.csv
[19:17:04] [KOSPI_index] Seed 42 started
[19:17:05] [KOSPI_index][seed=42] Training Autoformer...
[19:17:17] [KOSPI_index][seed=42] Done Autoformer | epochs=30 | best_val_loss=0.720978 | time=12.9s
[19:17:17] [KOSPI_index][seed=42] Training Informer...
[19:17:22] [KOSPI_index][seed=42] Done Informer | epochs=

,time,dataset,model,horizon,True_Volatility,Pred_Volatility,return_1_day
0,2022-08-23 00:00:00,KOSPI_index,Autoformer,1,1.174573,1.014601,-1.109068
1,2022-08-24 00:00:00,KOSPI_index,Autoformer,1,1.181606,1.301603,0.496023
2,2022-08-25 00:00:00,KOSPI_index,Autoformer,1,1.175803,1.363391,1.210647
3,2022-08-26 00:00:00,KOSPI_index,Autoformer,1,1.176170,1.119400,0.152069
4,2022-08-29 00:00:00,KOSPI_index,Autoformer,1,1.172896,0.993311,-2.206325
